# 05. 멀티 LLM 비교 분석

## 학습 목표
- GPT-4o-mini, Gemini Flash, Perplexity Sonar의 아키텍처 차이 이해
- 동일 프롬프트에 대한 엔진별 응답 패턴 분석
- 왜 엔진마다 다른 결과를 내는지 가설 수립
- 분석 프레임워크를 만들고 시각화

## 실험 설계
- 한국 로컬 비즈니스 관련 프롬프트 10개
- 3개 엔진 응답 비교 (샘플 데이터 하드코딩)
- 매장 수, 응답 구조, 설명 스타일 분석

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from collections import Counter

# 한글 폰트 설정 (Google Colab)
try:
    matplotlib.rcParams['font.family'] = 'NanumGothic'
except:
    pass  # 폰트 없으면 기본 폰트 사용

matplotlib.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. 현대 LLM 아키텍처 비교

### GPT-4o-mini (OpenAI)
- **아키텍처**: Decoder-only Transformer
- **특징**: GPT-4o의 경량 버전, 비용 효율적
- **학습 데이터**: 대규모 인터넷 텍스트 (cutoff 기반)
- **강점**: 범용 지식, 추론 능력, 지시 따르기
- **약점**: 학습 데이터 이후 정보 부재, 실시간 검색 없음

### Gemini Flash (Google)
- **아키텍처**: Mixture of Experts (MoE) 기반 Transformer
- **특징**: 빠른 추론 속도, 긴 컨텍스트 윈도우 (1M tokens)
- **학습 데이터**: Google 검색 인덱스 + 대규모 텍스트
- **강점**: 속도, 멀티모달, Google 생태계 연동
- **약점**: 비교적 최근 출시, 생태계 성숙도

### Perplexity Sonar
- **아키텍처**: 검색 증강 생성 (RAG 기반)
- **특징**: 실시간 웹 검색 + LLM 생성의 결합
- **학습 데이터**: 기본 LLM + 실시간 검색 결과
- **강점**: 최신 정보, 출처 제공, 사실 기반 응답
- **약점**: 검색 결과 품질에 의존, 창의적 생성 제한

In [ ]:
# 엔진별 특성 비교 테이블

comparison = {
    'Feature': [
        'Architecture', 'Knowledge Source', 'Real-time Search',
        'Response Speed', 'Context Window', 'Pricing (relative)',
        'Best For'
    ],
    'GPT-4o-mini': [
        'Decoder-only', 'Training data (cutoff)', 'No',
        'Fast', '128K tokens', 'Low',
        'General tasks'
    ],
    'Gemini Flash': [
        'MoE Transformer', 'Google index + training', 'Partial (Grounding)',
        'Very Fast', '1M tokens', 'Very Low',
        'Speed-critical tasks'
    ],
    'Perplexity Sonar': [
        'RAG (Search + LLM)', 'Real-time web search', 'Yes',
        'Medium', 'Varies', 'Medium',
        'Factual Q&A'
    ],
}

df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_string(index=False))

---
## 2. 실험 설계: 프롬프트 준비

한국 로컬 비즈니스 관련 프롬프트 10개를 준비한다.
이 프롬프트는 각 엔진의 로컬 지식, 최신성, 응답 스타일 차이를 드러낸다.

In [ ]:
# 실험 프롬프트 10개

prompts = [
    "강남에서 분위기 좋은 카페 3곳 추천해줘",
    "홍대 맛집 알려줘. 2만원 이하로 먹을 수 있는 곳",
    "판교 테크노밸리 근처 점심 맛집 추천",
    "성수동 데이트 코스 추천해줘",
    "을지로 노포 맛집 베스트 3",
    "이태원 브런치 카페 추천",
    "여의도 직장인 점심 맛집",
    "연남동 비건 레스토랑 있어?",
    "잠실 롯데월드몰 근처 저녁 식사 추천",
    "망원동 베이커리 카페 추천해줘",
]

print("실험 프롬프트 목록:")
for i, p in enumerate(prompts, 1):
    print(f"  {i:>2}. {p}")

print(f"\n총 {len(prompts)}개 프롬프트 x 3개 엔진 = {len(prompts)*3}개 응답 분석")

---
## 3. 샘플 응답 데이터

실제 API 호출 코드는 placeholder로 두고,
분석 프레임워크가 동작하도록 **샘플 응답 데이터를 하드코딩**한다.

### API 호출 Placeholder

In [ ]:
# === API 호출 Placeholder ===
# 실제 사용 시 아래 함수를 구현하세요

def call_gpt4o_mini(prompt):
    """
    GPT-4o-mini API 호출 (placeholder)
    
    실제 구현:
    from openai import OpenAI
    client = OpenAI(api_key="YOUR_KEY")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content
    """
    pass

def call_gemini_flash(prompt):
    """
    Gemini Flash API 호출 (placeholder)
    
    실제 구현:
    import google.generativeai as genai
    genai.configure(api_key="YOUR_KEY")
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)
    return response.text
    """
    pass

def call_perplexity_sonar(prompt):
    """
    Perplexity Sonar API 호출 (placeholder)
    
    실제 구현:
    from openai import OpenAI
    client = OpenAI(api_key="YOUR_KEY", base_url="https://api.perplexity.ai")
    response = client.chat.completions.create(
        model="sonar",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content
    """
    pass

print("API placeholder 함수 정의 완료")
print("실제 사용 시 각 함수에 API 키와 호출 로직을 구현하세요")

In [ ]:
# === 샘플 응답 데이터 (하드코딩) ===
# 실제 API 응답을 시뮬레이션한 데이터

sample_responses = {
    "강남에서 분위기 좋은 카페 3곳 추천해줘": {
        "gpt4o_mini": {
            "stores": ["카페 온도", "테라로사 강남점", "블루보틀 삼성점"],
            "response_length": 312,
            "has_address": True,
            "has_price": False,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["테라로사 강남점", "카페 드 마고", "앤트러사이트 강남"],
            "response_length": 287,
            "has_address": True,
            "has_price": True,
            "style": "structured",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["카페 레이어드 강남점", "하프커피", "카멜커피"],
            "response_length": 445,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 4,
        },
    },
    "홍대 맛집 알려줘. 2만원 이하로 먹을 수 있는 곳": {
        "gpt4o_mini": {
            "stores": ["연남서가", "우래옥 홍대점", "돈부리 마루"],
            "response_length": 356,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["미미네", "동경규동", "홍대 돈카츠 히레"],
            "response_length": 298,
            "has_address": True,
            "has_price": True,
            "style": "list",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["키친노을", "연남서가", "동경규동", "풍년쌀농산"],
            "response_length": 523,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 5,
        },
    },
    "판교 테크노밸리 근처 점심 맛집 추천": {
        "gpt4o_mini": {
            "stores": ["백소정 판교점", "한촌설렁탕", "이디야 판교점"],
            "response_length": 289,
            "has_address": False,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["정돈 판교점", "오봉집 판교", "서울깍두기"],
            "response_length": 342,
            "has_address": True,
            "has_price": True,
            "style": "structured",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["맛찬들 왕소금구이", "정돈 판교점", "봉피양"],
            "response_length": 478,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 3,
        },
    },
    "성수동 데이트 코스 추천해줘": {
        "gpt4o_mini": {
            "stores": ["대림창고", "성수연방", "카페 할아버지공장", "오르에르"],
            "response_length": 420,
            "has_address": True,
            "has_price": False,
            "style": "narrative",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["성수연방", "LCDC 서울", "카페 봇", "뚝섬유원지"],
            "response_length": 389,
            "has_address": True,
            "has_price": False,
            "style": "structured",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["성수연방", "대림창고", "어니언 성수", "서울숲"],
            "response_length": 567,
            "has_address": True,
            "has_price": False,
            "style": "factual",
            "sources_cited": 6,
        },
    },
    "을지로 노포 맛집 베스트 3": {
        "gpt4o_mini": {
            "stores": ["을지면옥", "양미옥", "을지OB베어"],
            "response_length": 278,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["을지면옥", "무교동 북어국집", "안성집"],
            "response_length": 315,
            "has_address": True,
            "has_price": True,
            "style": "list",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["을지면옥", "양미옥", "무교동 북어국집"],
            "response_length": 401,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 4,
        },
    },
    "이태원 브런치 카페 추천": {
        "gpt4o_mini": {
            "stores": ["라 카페테리아", "부츠 이태원", "클리프 이태원"],
            "response_length": 302,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["타르틴 베이커리", "부츠 이태원", "포스트낵"],
            "response_length": 267,
            "has_address": True,
            "has_price": True,
            "style": "list",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["부츠 이태원", "하디스 브런치", "르 스타일"],
            "response_length": 412,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 3,
        },
    },
    "여의도 직장인 점심 맛집": {
        "gpt4o_mini": {
            "stores": ["더현대 서울 푸드코트", "여의도 쌀국수", "마포갈매기"],
            "response_length": 334,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["한촌설렁탕 여의도", "미진 여의도", "봉피양 여의도"],
            "response_length": 290,
            "has_address": True,
            "has_price": True,
            "style": "structured",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["한촌설렁탕 여의도", "더현대 서울 식당가", "진주회관"],
            "response_length": 489,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 5,
        },
    },
    "연남동 비건 레스토랑 있어?": {
        "gpt4o_mini": {
            "stores": ["플랜튜드", "오세게"],
            "response_length": 245,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["플랜튜드", "초식공간", "발우공양"],
            "response_length": 310,
            "has_address": True,
            "has_price": True,
            "style": "list",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["플랜튜드", "오세게", "초식공간"],
            "response_length": 456,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 4,
        },
    },
    "잠실 롯데월드몰 근처 저녁 식사 추천": {
        "gpt4o_mini": {
            "stores": ["몽탄 잠실점", "파이어벨 잠실", "한촌설렁탕 잠실"],
            "response_length": 356,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["롯데월드몰 다이닝", "피에스타 데 토로", "부촌 잠실"],
            "response_length": 275,
            "has_address": False,
            "has_price": True,
            "style": "list",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["몽탄 잠실점", "롯데월드몰 맛집", "마장동 소고기"],
            "response_length": 501,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 5,
        },
    },
    "망원동 베이커리 카페 추천해줘": {
        "gpt4o_mini": {
            "stores": ["밀도 망원점", "에이프릴", "나이스웨더"],
            "response_length": 298,
            "has_address": True,
            "has_price": True,
            "style": "descriptive",
            "sources_cited": 0,
        },
        "gemini_flash": {
            "stores": ["밀도 망원점", "르뺑 뀌오띠디앙", "오월의 종"],
            "response_length": 256,
            "has_address": True,
            "has_price": True,
            "style": "list",
            "sources_cited": 0,
        },
        "perplexity_sonar": {
            "stores": ["밀도 망원점", "에이프릴", "르뺑 뀌오띠디앙", "나이스웨더"],
            "response_length": 534,
            "has_address": True,
            "has_price": True,
            "style": "factual",
            "sources_cited": 6,
        },
    },
}

print(f"샘플 데이터: {len(sample_responses)}개 프롬프트 x 3개 엔진")

---
## 4. 응답 분석 프레임워크

In [ ]:
# 분석 데이터 구조화

engines = ['gpt4o_mini', 'gemini_flash', 'perplexity_sonar']
engine_labels = ['GPT-4o-mini', 'Gemini Flash', 'Perplexity Sonar']

# DataFrame으로 변환
rows = []
for prompt, responses in sample_responses.items():
    for engine in engines:
        r = responses[engine]
        rows.append({
            'prompt': prompt,
            'engine': engine,
            'n_stores': len(r['stores']),
            'stores': r['stores'],
            'response_length': r['response_length'],
            'has_address': r['has_address'],
            'has_price': r['has_price'],
            'style': r['style'],
            'sources_cited': r['sources_cited'],
        })

df = pd.DataFrame(rows)
print(f"분석 데이터: {len(df)}개 레코드")
print(f"\n엔진별 평균 통계:")
print(df.groupby('engine')[['n_stores', 'response_length', 'sources_cited']].mean().round(1))

In [ ]:
# 매장 중복률 분석

def calculate_overlap(responses, engine1, engine2):
    """두 엔진 간 추천 매장 중복률 계산"""
    overlaps = []
    for prompt, resp in responses.items():
        stores1 = set(resp[engine1]['stores'])
        stores2 = set(resp[engine2]['stores'])
        if stores1 and stores2:
            overlap = len(stores1 & stores2) / len(stores1 | stores2)
            overlaps.append(overlap)
    return np.mean(overlaps)


# 엔진 쌍별 중복률
pairs = [
    ('gpt4o_mini', 'gemini_flash'),
    ('gpt4o_mini', 'perplexity_sonar'),
    ('gemini_flash', 'perplexity_sonar'),
]

print("엔진 쌍별 매장 추천 중복률 (Jaccard Similarity):")
overlap_matrix = np.zeros((3, 3))
for i in range(3):
    overlap_matrix[i][i] = 1.0

for e1, e2 in pairs:
    overlap = calculate_overlap(sample_responses, e1, e2)
    i = engines.index(e1)
    j = engines.index(e2)
    overlap_matrix[i][j] = overlap
    overlap_matrix[j][i] = overlap
    print(f"  {e1} vs {e2}: {overlap:.2%}")

print(f"\n→ 엔진마다 추천하는 매장이 상당히 다르다!")

---
## 5. 가설 수립: 왜 엔진마다 다른 매장을 추천하는가?

### 가설 1: 학습 데이터 차이
- GPT-4o-mini: 영어 중심 데이터가 많아 한국 로컬 정보가 상대적으로 부족할 수 있음
- Gemini Flash: Google 한국 검색 데이터 반영, 네이버/카카오맵 리뷰와 다를 수 있음
- Perplexity: 실시간 검색이므로 현재 시점의 인기 매장 반영

### 가설 2: 검색 기반 vs 지식 기반
- GPT, Gemini: 학습 시점의 **정적 지식** → 폐업 매장 추천 가능
- Perplexity: 실시간 **검색 결과** → 현재 영업 중인 매장

### 가설 3: Recency Bias
- 검색 기반 엔진은 최근 리뷰/블로그가 많은 매장을 우선
- 지식 기반 엔진은 오래된 유명 맛집도 동등하게 추천

### 가설 4: 응답 스타일 차이
- GPT: 서술적 (descriptive), 분위기/맛 설명에 집중
- Gemini: 구조적 (structured), 리스트형 정보 제공
- Perplexity: 사실적 (factual), 출처와 함께 근거 제시

---
## 6. 분석 결과 시각화

In [ ]:
# 시각화 1: 엔진별 응답 길이 분포

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['#4285F4', '#34A853', '#EA4335']  # Google 색상

# 1. 응답 길이 비교
ax = axes[0]
for i, engine in enumerate(engines):
    lengths = df[df['engine'] == engine]['response_length']
    ax.bar(i, lengths.mean(), yerr=lengths.std(), color=colors[i],
           alpha=0.7, capsize=5, label=engine_labels[i])
ax.set_xticks(range(3))
ax.set_xticklabels(engine_labels, fontsize=9)
ax.set_ylabel('Response Length (chars)')
ax.set_title('Average Response Length', fontsize=12)
ax.grid(axis='y', alpha=0.3)

# 2. 추천 매장 수 비교
ax = axes[1]
for i, engine in enumerate(engines):
    n_stores = df[df['engine'] == engine]['n_stores']
    ax.bar(i, n_stores.mean(), yerr=n_stores.std(), color=colors[i],
           alpha=0.7, capsize=5)
ax.set_xticks(range(3))
ax.set_xticklabels(engine_labels, fontsize=9)
ax.set_ylabel('Number of Stores')
ax.set_title('Average Stores Recommended', fontsize=12)
ax.grid(axis='y', alpha=0.3)

# 3. 출처 인용 수
ax = axes[2]
for i, engine in enumerate(engines):
    sources = df[df['engine'] == engine]['sources_cited']
    ax.bar(i, sources.mean(), yerr=sources.std(), color=colors[i],
           alpha=0.7, capsize=5)
ax.set_xticks(range(3))
ax.set_xticklabels(engine_labels, fontsize=9)
ax.set_ylabel('Sources Cited')
ax.set_title('Average Sources Cited', fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Perplexity Sonar: 가장 긴 응답, 가장 많은 매장, 출처 인용")
print("Gemini Flash: 간결한 응답, 구조적")
print("GPT-4o-mini: 중간 길이, 서술적 스타일")

In [ ]:
# 시각화 2: 매장 중복률 히트맵

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 중복률 히트맵
ax = axes[0]
im = ax.imshow(overlap_matrix, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(engine_labels, fontsize=10)
ax.set_yticklabels(engine_labels, fontsize=10)
ax.set_title('Store Recommendation Overlap\n(Jaccard Similarity)', fontsize=12)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{overlap_matrix[i,j]:.2f}',
                ha='center', va='center', fontsize=12,
                color='white' if overlap_matrix[i,j] > 0.5 else 'black')
plt.colorbar(im, ax=ax)

# 응답 스타일 분포
ax = axes[1]
style_counts = df.groupby(['engine', 'style']).size().unstack(fill_value=0)
style_counts = style_counts.reindex(engines)
style_counts.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_xticklabels(engine_labels, rotation=0, fontsize=10)
ax.set_ylabel('Count')
ax.set_title('Response Style Distribution', fontsize=12)
ax.legend(title='Style', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("엔진 간 매장 추천 중복률이 낮다 → 각 엔진이 다른 소스/기준으로 추천")
print("GPT: descriptive/narrative, Gemini: structured/list, Perplexity: factual")

In [ ]:
# 시각화 3: 프롬프트별 엔진 간 상세 비교

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 프롬프트별 응답 길이
ax = axes[0]
prompt_labels = [p[:15] + '...' for p in prompts]
x = np.arange(len(prompts))
width = 0.25

for i, engine in enumerate(engines):
    lengths = df[df['engine'] == engine]['response_length'].values
    ax.bar(x + i*width, lengths, width, label=engine_labels[i],
           color=colors[i], alpha=0.7)

ax.set_xticks(x + width)
ax.set_xticklabels(prompt_labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Response Length (chars)')
ax.set_title('Response Length by Prompt', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 프롬프트별 추천 매장 수
ax = axes[1]
for i, engine in enumerate(engines):
    n_stores = df[df['engine'] == engine]['n_stores'].values
    ax.bar(x + i*width, n_stores, width, label=engine_labels[i],
           color=colors[i], alpha=0.7)

ax.set_xticks(x + width)
ax.set_xticklabels(prompt_labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Stores')
ax.set_title('Stores Recommended by Prompt', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 시각화 4: 전체 매장 등장 빈도 분석

# 모든 매장 수집
all_stores = {engine: [] for engine in engines}
for prompt, responses in sample_responses.items():
    for engine in engines:
        all_stores[engine].extend(responses[engine]['stores'])

# 엔진별 고유 매장 수
unique_stores = {engine: set(stores) for engine, stores in all_stores.items()}
all_unique = set()
for stores in unique_stores.values():
    all_unique.update(stores)

print(f"전체 고유 매장 수: {len(all_unique)}")
for engine, stores in unique_stores.items():
    print(f"  {engine}: {len(stores)}개")

# 엔진 간 공통 매장
common_all = unique_stores['gpt4o_mini'] & unique_stores['gemini_flash'] & unique_stores['perplexity_sonar']
print(f"\n3개 엔진 모두 추천한 매장 ({len(common_all)}개):")
for store in common_all:
    print(f"  - {store}")

# 한 엔진만 추천한 매장
for engine in engines:
    others = set()
    for e in engines:
        if e != engine:
            others.update(unique_stores[e])
    exclusive = unique_stores[engine] - others
    print(f"\n{engine}만 추천한 매장 ({len(exclusive)}개):")
    for store in list(exclusive)[:5]:
        print(f"  - {store}")

In [ ]:
# Venn Diagram 스타일 시각화 (matplotlib로 근사)

fig, ax = plt.subplots(figsize=(8, 6))

# 집합 크기
sets = {
    'GPT': unique_stores['gpt4o_mini'],
    'Gemini': unique_stores['gemini_flash'],
    'Perplexity': unique_stores['perplexity_sonar'],
}

# 교집합 크기 계산
gpt_only = len(sets['GPT'] - sets['Gemini'] - sets['Perplexity'])
gem_only = len(sets['Gemini'] - sets['GPT'] - sets['Perplexity'])
ppl_only = len(sets['Perplexity'] - sets['GPT'] - sets['Gemini'])
gpt_gem = len((sets['GPT'] & sets['Gemini']) - sets['Perplexity'])
gpt_ppl = len((sets['GPT'] & sets['Perplexity']) - sets['Gemini'])
gem_ppl = len((sets['Gemini'] & sets['Perplexity']) - sets['GPT'])
all_three = len(sets['GPT'] & sets['Gemini'] & sets['Perplexity'])

# 막대 그래프로 표현
categories = [
    f'GPT only\n({gpt_only})',
    f'Gemini only\n({gem_only})',
    f'Pplx only\n({ppl_only})',
    f'GPT+Gemini\n({gpt_gem})',
    f'GPT+Pplx\n({gpt_ppl})',
    f'Gemini+Pplx\n({gem_ppl})',
    f'All three\n({all_three})',
]
values = [gpt_only, gem_only, ppl_only, gpt_gem, gpt_ppl, gem_ppl, all_three]
bar_colors = ['#4285F4', '#34A853', '#EA4335', '#7B68EE', '#FF6347', '#FFD700', '#333333']

bars = ax.bar(range(len(categories)), values, color=bar_colors, alpha=0.8)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylabel('Number of Unique Stores')
ax.set_title('Store Recommendation Overlap Analysis', fontsize=13)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n총 {len(all_unique)}개 고유 매장 중 3개 엔진 공통은 {all_three}개 ({all_three/len(all_unique):.0%})")
print("→ 각 엔진이 상당히 다른 매장을 추천한다!")

In [ ]:
# 종합 분석 요약

print("=" * 60)
print("          멀티 LLM 응답 비교 분석 요약")
print("=" * 60)

print("\n[1] 응답 스타일")
print("  - GPT-4o-mini:      서술적, 분위기/경험 중심 설명")
print("  - Gemini Flash:     구조적/리스트, 핵심 정보 간결 전달")
print("  - Perplexity Sonar: 사실 기반, 출처 인용, 상세 정보")

print("\n[2] 정보 정확도 (추정)")
print("  - GPT-4o-mini:      학습 데이터 기반 → 폐업 매장 추천 위험")
print("  - Gemini Flash:     Google 데이터 → 비교적 최신")
print("  - Perplexity Sonar: 실시간 검색 → 가장 최신")

print("\n[3] 매장 추천 패턴")
print(f"  - 3개 엔진 공통 추천: 전체의 약 {all_three/len(all_unique):.0%} (유명 매장)")
print(f"  - 엔진별 독자 추천: 각각 상당수의 고유 매장 보유")

print("\n[4] 실용적 시사점")
print("  - 로컬 비즈니스 추천: Perplexity(검색 기반)이 가장 적합")
print("  - 일반적 정보 정리: GPT 또는 Gemini가 효율적")
print("  - 최상의 결과: 여러 엔진 결과를 교차 검증")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 추가 분석 지표 구현

아래 분석 지표를 추가로 구현하고 시각화하세요:

1. **정보 밀도** (Information Density): 응답 길이 대비 매장 수 비율
   - `info_density = n_stores / response_length * 1000`
   - 높을수록 간결하게 많은 정보를 전달

2. **정보 완성도** (Completeness Score): 주소 + 가격 정보 포함 비율
   - `completeness = (has_address + has_price) / 2`
   - 1에 가까울수록 완전한 정보 제공

3. 위 두 지표를 엔진별로 비교하는 차트를 그리세요

In [ ]:
# TODO:
# 1. df에 info_density, completeness 컬럼 추가
# 2. 엔진별 평균 info_density, completeness 계산
# 3. 나란히 비교하는 bar chart 그리기
# 4. 어떤 엔진이 가장 효율적인 정보 전달을 하는지 분석


---
## 핵심 정리

| 관점 | GPT-4o-mini | Gemini Flash | Perplexity Sonar |
|------|------------|-------------|------------------|
| 아키텍처 | Decoder-only | MoE Transformer | RAG (Search + LLM) |
| 지식 소스 | 학습 데이터 (cutoff) | Google 인덱스 | 실시간 웹 검색 |
| 응답 스타일 | 서술적, 상세 | 구조적, 간결 | 사실적, 출처 제공 |
| 최신성 | 학습 시점까지 | 비교적 최신 | 가장 최신 |
| 매장 중복률 | 낮음 | 낮음 | 낮음 |
| 적합 용도 | 일반 추천, 분위기 | 빠른 정보, 리스트 | 최신 정보, 사실 확인 |

### 핵심 인사이트
1. 동일 프롬프트에도 엔진마다 **매우 다른 매장**을 추천한다
2. 차이의 주요 원인: **학습 데이터**, **검색 vs 지식**, **recency bias**
3. 로컬 비즈니스 추천에는 **검색 기반 엔진(Perplexity)**이 유리
4. 최상의 결과: **여러 엔진의 교차 검증**